In [10]:
!pip install -q datasets==3.6.0 librosa soundfile

In [6]:
from datasets import load_dataset

ds = load_dataset("cais/mmlu", "all")

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [7]:
ds = ds["auxiliary_train"]
ds

Dataset({
    features: ['question', 'subject', 'choices', 'answer'],
    num_rows: 99842
})

In [8]:
from datasets import load_from_disk
ref_ds = load_from_disk(os.environ.get("MMLU_REF_DS_PATH", "/path/to/your/dataset/MMLU_FULL/full_5k_hf_format.v1"))
ref_ds

Loading dataset from disk:   0%|          | 0/19 [00:00<?, ?it/s]

Dataset({
    features: ['key', 'question_id', 'question', 'options', 'answer', 'answer_index', 'question_for_tts', 'prompt_for_tts', 'audio', 'tts_wer', 'tts_retries', 'tts_wer_avg', 'category', 'prompt_for_tts_tn'],
    num_rows: 5000
})

In [11]:
ref_ds[0]

{'key': 'mmlu@en_10000',
 'question_id': 10000,
 'question': 'Hume describes reason as:',
 'options': ['cool and disengaged.',
  'the source of all moral actions.',
  'the first spring or impulse to desire.',
  'all of the above.'],
 'answer': 'cool and disengaged.',
 'answer_index': 0,
 'question_for_tts': 'Hume describes reason as:',
 'prompt_for_tts': 'Hume describes reason as:\n\nOption A: cool and disengaged.\nOption B: the source of all moral actions.\nOption C: the first spring or impulse to desire.\nOption D: all of the above.',
 'audio': {'path': 'mmlu@en_10000.wav',
  'array': array([ 1.22070312e-04,  1.22070312e-04,  0.00000000e+00, ...,
         -6.10351562e-05, -9.15527344e-05, -2.44140625e-04], shape=(454656,)),
  'sampling_rate': 44100},
 'tts_wer': 0.03125,
 'tts_retries': 1,
 'tts_wer_avg': 0.0703125,
 'category': 'philosophy',
 'prompt_for_tts_tn': 'hume describes reason as option a cool and disengaged option b the source of all moral actions option c the first spring

In [12]:
ds = ds.remove_columns(['subject'])

In [15]:
counter = 0
def map_func(item):
    global counter
    counter += 1
    return {
        'key': f'mmlu@{counter}',
        'question_id': counter,
        'question': item["question"],
        'audio': None,
        'options': item["choices"],
        'answer_index': item["answer"]
    }
ds = ds.map(map_func)

Map:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [18]:
ds = ds.remove_columns(['choices', 'answer'])

In [19]:
def process_map(question):
    # For Question:
    # - _{2,}\. 以下划线结尾的，修改为 what?
    # - _{2,} 出现一次，修改为 what
    # - _{2,} 出现多次，修改为 first blank, second blank ...
    import re
    
    # Pattern for underscore blanks: two or more underscores
    blank_pattern = re.compile(r'_{2,}')
    # Pattern for underscore blanks at the end of a sentence
    blank_end_pattern = re.compile(r'_{2,}\.$')
    
    # Replace underscore blanks at the end with "what?"
    question = blank_end_pattern.sub(' what?', question)
    
    # Find all remaining blanks
    blanks = blank_pattern.findall(question)
    
    if len(blanks) == 1:
        # If there's only one blank, replace it with "what"
        question = blank_pattern.sub(' what ', question)
    elif len(blanks) > 1:
        # If there are multiple blanks, replace them with "first blank", "second blank", etc.
        for i, _ in enumerate(blanks):
            ordinal = ["first", "second", "third", "fourth", "fifth", "sixth", "seventh", "eighth", "ninth", "tenth"][i % 10]
            # Replace only the first occurrence after each iteration
            question = blank_pattern.sub(f' {ordinal} blank ', question, 1)
    
    return {"question_for_tts": question}


ds = ds.map(process_map, input_columns=["question"])

Map:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [20]:
def add_options(question, options):
    # template to add options
    prompt_for_tts = f"{question}\n\nOption A: {options[0]}\nOption B: {options[1]}\nOption C: {options[2]}\nOption D: {options[3]}"
    return {"prompt_for_tts": prompt_for_tts}


ds = ds.map(add_options, input_columns=["question_for_tts", "options"])

Map:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [21]:
ds[0]

{'question': "Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thinking that Brooks was Adams, Davis shot at Brooks. The shot missed Brooks but wounded Case, who was some distance away. Davis had not seen Case. In a prosecution under a statute that proscribes any attempt to commit murder, the district attorney should indicate that the intended victim(s) was/were",
 'key': 'mmlu@1',
 'question_id': 1,
 'audio': None,
 'options': ['Adams only.', 'Brooks only.', 'Case only.', 'Adams and Brooks'],
 'answer_index': 1,
 'question_for_tts': "Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thinking that Brooks was Adams, Davis shot at Brooks. The shot missed Brooks but wounded Case, who was some distance away. Davis had not seen Case. In a prosecution under a statute that proscribes any attempt to commit murder, the district attorney should indicate that the inte

In [23]:
ref_prompt_for_tts = ref_ds["prompt_for_tts"]
ref_prompt_for_tts = [len(t) for t in ref_prompt_for_tts]
import numpy as np

np.mean(ref_prompt_for_tts), np.percentile(ref_prompt_for_tts, 90), np.percentile(ref_prompt_for_tts, 95), np.percentile(ref_prompt_for_tts, 99), np.max(ref_prompt_for_tts)

(np.float64(418.2836),
 np.float64(862.1000000000004),
 np.float64(1232.1000000000004),
 np.float64(1822.0200000000004),
 np.int64(2769))

In [27]:
ref_prompt_for_tts = ds["prompt_for_tts"]
ref_prompt_for_tts = [len(t) for t in ref_prompt_for_tts]
ref_prompt_for_tts = [t for t in ref_prompt_for_tts if t < 2000]
import numpy as np

np.mean(ref_prompt_for_tts), np.percentile(ref_prompt_for_tts, 90), np.percentile(ref_prompt_for_tts, 95), np.percentile(ref_prompt_for_tts, 99), np.max(ref_prompt_for_tts), len(ref_prompt_for_tts)

(np.float64(1276.8982171987882),
 np.float64(1906.0),
 np.float64(1953.0),
 np.float64(1990.0),
 np.int64(1999),
 68656)

In [31]:
ds = ds.filter(lambda x: len(x) <= 2000, input_columns=["prompt_for_tts"])

Filter:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [33]:
ds[0]

{'question': "Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thinking that Brooks was Adams, Davis shot at Brooks. The shot missed Brooks but wounded Case, who was some distance away. Davis had not seen Case. In a prosecution under a statute that proscribes any attempt to commit murder, the district attorney should indicate that the intended victim(s) was/were",
 'key': 'mmlu@1',
 'question_id': 1,
 'audio': None,
 'options': ['Adams only.', 'Brooks only.', 'Case only.', 'Adams and Brooks'],
 'answer_index': 1,
 'question_for_tts': "Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thinking that Brooks was Adams, Davis shot at Brooks. The shot missed Brooks but wounded Case, who was some distance away. Davis had not seen Case. In a prosecution under a statute that proscribes any attempt to commit murder, the district attorney should indicate that the inte

In [32]:
ds.save_to_disk(os.environ.get("TEXTONLY_DS_PATH", "/path/to/your/dataset/MMLU_FULL/full_train_hf_format.textonly.v0"))

Saving the dataset (0/1 shards):   0%|          | 0/68739 [00:00<?, ? examples/s]